In [ ]:
import os
from datetime import datetime, timedelta
import glob
from multiprocessing import Pool
import pandas as pd
import seaborn as sns
from cartopy.crs import NorthPolarStereo
from cartopy.feature import LAND, COASTLINE
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import numpy as np
from cmocean import cm

DAY_SECONDS = 24 * 60 * 60

In [ ]:
def decorate_map(ax, map_extent, crs, title):
    ax.add_feature(LAND)
    ax.add_feature(COASTLINE)
    ax.set_extent(map_extent, crs=crs)

srs_dst = NorthPolarStereo(central_longitude=-45, true_scale_latitude=60)
map_extent = [-2300000, 300000, -1000000, 2100000]

In [ ]:
odir = '../tuning_paper_figures'
idir = '../music_matrix/cfg04_m20'

# best before tuning (default)
cfg01_indices = [21]

cfg01_files = []
pfiles = []
for i in cfg01_indices:
    pfile = f'../music_matrix/cfg01_m20/mat{i:02}_pairs.npz'
    pfiles.append(pfile)
    cfg01_files.append(pfile)
# best after tuning
cfg04_file = f'{idir}/mat03_pairs.npz'
pfiles.append(cfg04_file)

rgps_file = '../rgps_csv/w07_may_pairs.npz'
pfiles += [rgps_file]


In [ ]:
names = ['div', 'she']
cmaps = [cm.balance, 'plasma_r']
full_names = [
    'Divergence',
    'Shear',
]
vmax = 0.1
vmins = [-vmax, 0]
period_len = timedelta(4)
force = True
date1_list = [
    datetime(2007,1,25),
    datetime(2007,2,3),
    datetime(2007,2,15)
]

pairs = {}
defor = {}
for pfile in pfiles:
    dfile = pfile.replace('_pairs.npz', '_defor.npz')
    pairs[pfile] = np.load(pfile, allow_pickle=True)['pairs']
    defor[pfile] = np.load(dfile, allow_pickle=True)['defor']


In [ ]:
def inset_colobar_axis(ax, trp, title):
    bgax = inset_axes(ax, '99%', '15%', loc='lower center')
    bgax.set_facecolor((1,1,1,0.95))
    bgax.spines['top'].set_visible(False)
    bgax.spines['right'].set_visible(False)
    bgax.spines['bottom'].set_visible(False)
    bgax.spines['left'].set_visible(False)
    bgax.tick_params(colors=(0,0,0,0), )
    cbar_ax = inset_axes(bgax, '85%', '15%', loc='upper center')
    cbar = fig.colorbar(trp, cax=cbar_ax, orientation='horizontal')
    cbar.set_label(title, fontsize=14)
    cbar.ax.tick_params(labelsize=14)

sources = ['neXtSIM control', 'neXtSIM optimal', 'RGPS']

for cfg01_file in cfg01_files:
    panel_size = 5
    for date1 in date1_list:
        fig, axs = plt.subplots(2, 3, figsize=(3*panel_size, 2.3*panel_size), subplot_kw={'projection': srs_dst})
        ofile = f'{odir}/fig00_best_defor_maps_{os.path.basename(cfg01_file)}_{date1.strftime("%Y%m%d")}.png'
        pfiles = [cfg01_file, cfg04_file, rgps_file]
        for i, (pfile, src) in enumerate(zip(pfiles, sources)):
            for p, d in zip(pairs[pfile], defor[pfile]):
                if p is None or d is None:
                    continue
                if date1 <= p.d1 < (date1 + period_len):
                    trp0 = axs[0, i].tripcolor(p.x1, p.y1, d.e1 * DAY_SECONDS, triangles=p.t, vmin=-vmax, vmax=vmax, cmap=cm.balance, mask=~p.g)
                    trp1 = axs[1, i].tripcolor(p.x1, p.y1, d.e2 * DAY_SECONDS, triangles=p.t, vmin=0, vmax=vmax, cmap='plasma_r', mask=~p.g)
        for ax in axs.flat:
            decorate_map(ax, map_extent, srs_dst, src)

        inset_colobar_axis(axs[0,1], trp0, 'Divergence, 1/day')
        inset_colobar_axis(axs[1,1], trp1, 'Shear, 1/day')
        axs[0,0].set_title('neXtSIM default', fontsize=14)
        axs[0,1].set_title('neXtSIM optimal', fontsize=14)
        axs[0,2].set_title('RGPS', fontsize=14)
        plt.tight_layout()
        plt.savefig(ofile, dpi=150, bbox_inches='tight', pad_inches=0.1)
        plt.close()
        print(ofile)